This is a GUI for sentiment analysis using HuggingFace's transformer model.

This provides a UI for the user to perform sentiment analysis, either through text input or data from a CSV files. Utilises the cardiffnlp/twitter-roberta-base-sentiment-latest model from HuggingFace to make prediction of the sentiments and then displays the results
After displaying the result, it provides means for user to save the new results locally as a .CSV file, if provided through the file processing tab"

Importing the necessary python modules

In [1]:
import tkinter as tk
from tkinter import *
from tkinter import ttk, filedialog

import pandas as pd
import numpy as np
import seaborn as sns

import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm.notebook import tqdm

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


load sentiment analysis model from Hugging Face

In [2]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Defining the app window, Title assignment, and Size definition

In [3]:
window = Tk()
window.title("Sentiment Analysis GUI")
window.geometry("1000x650")

''

Tab Layout definition

In [4]:
style = ttk.Style()
style.configure("TNotebook.Tab", padding=[20, 10])

tab_control = ttk.Notebook(window)
tab1 = ttk.Frame(tab_control)
tab2 = ttk.Frame(tab_control)
tab3 = ttk.Frame(tab_control)

Add tab to notebook

In [5]:
tab_control.add(tab1, text="Text Analysis")
tab_control.add(tab2, text="File Processor")
tab_control.add(tab3, text="Help")

tab_control.pack(expand=1,fill='both')

Tab for Text Analysis

In [6]:
label1 = Label(tab1,text="Use here to analyse individual reviews", font=("Comic Sans MS",12), padx=10,pady=5)
label2 = Label(tab2,text="Use here to analyse CSV file", font=("Comic Sans MS",12), padx=5,pady=5)
label3 = Label(tab3,text="Instructions on how to use the app", font=("Comic Sans MS",12), padx=5,pady=5)

label1.grid(column=0, row=0)
label2.grid(column=0, row=0)
label3.grid(column=0, row=0)

Perform tokenization, POS tagging, Sentiment analysis of text/sentences inputted into the textbox. Also cleans/clear both inputted texts as well as results
Retrieves text entered into the input widget, tokenizes it, performs POS tagging, processes it through the sentiment analysis model, and displays the predicted sentiment.
Updates the result in the GUI's output label

Tab 1 Function for tokenisation of NLP

In [7]:
def get_tokens():
    global raw_text
    raw_text = str(raw_entry.get())
    global tokens
    tokens = tokenizer.tokenize(raw_text)
    result = '\nTokens: {}'.format(tokens)
    # Insert into display
    tab1_display.insert(tk.END,result)

Tab 1 Function for POS Tagging of NLP

In [8]:
def get_pos_tags():
    pos_tags = nltk.pos_tag(tokens)
    result = '\nPOS Tagger: {}'.format(pos_tags)
    # Insert into display
    tab1_display.insert(tk.END,result)

Tab 1 Function for Sentiment Analysis

In [9]:
def get_sentiment():
    raw_text = tab1_display.get("1.0",tk.END)
    encoded_text = tokenizer(raw_text, return_tensors="pt")
    output = model(**encoded_text)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
        "Negative" : scores[0],
        "Neutral" : scores[1],
        "Positive" : scores[2]
    }
    for key, value in scores_dict.items():
        tab1_display.insert(tk.END, f"\n{key}: {value:.4f}\n")

Tab 1 Function to clear/clean input text field

In [10]:
def clear_text_entry():
    entry1.delete(0,END)

Tab 1 Function to clear/clean display result/results

In [11]:
def clear_display_result():
    tab1_display.delete('1.0',END)

In [12]:
l1 = Label(tab1,text="Enter Text to Analyse", font=("Comic Sans MS", 14))
l1.grid(row=3, column=0)
          
raw_entry = StringVar()
entry1 = Entry(tab1, textvariable=raw_entry,width=50, font=("Arial", 14))
entry1.grid(row=3,column=1, pady=10)

Tab 1 Buttons

In [13]:
button1 = Button(tab1,text="Tokenize",width=15,bg="cornflower blue",fg="snow",command=get_tokens)
button1.grid(row=6,column=0, 
             padx=2,pady=10)
button1.config(font=("Comic Sans MS", 15))

button2 = Button(tab1,text="POS Tagger",width=15,bg="cornflower blue",fg="snow",command=get_pos_tags)
button2.grid(row=6,column=1, 
             padx=(0, 250),pady=10)
button2.config(font=("Comic Sans MS", 15))

button3 = Button(tab1,text="Sentiment",width=15,bg="cornflower blue",fg="snow",command=get_sentiment)
button3.grid(row=6,column=1, 
             padx=2,pady=10,
            sticky="e")
button3.config(font=("Comic Sans MS", 15))

button4 = Button(tab1,text="Clear Entry Text",width=15,bg="cornflower blue",fg="snow",command=clear_text_entry)
button4.grid(row=7,column=0, 
             padx=2,pady=10)
button4.config(font=("Comic Sans MS", 15))

button5 = Button(tab1,text="Clear Result",width=15,bg="cornflower blue",fg="snow",command=clear_display_result)
button5.grid(row=7,column=1, 
             padx=2,pady=10, 
             sticky="e")
button5.config(font=("Comic Sans MS", 15))

Tab 1 Display screen for Result

In [14]:
tab1_display = Text(tab1, height=10, width=80)
tab1_display.grid(row=8, column=0, columnspan=4,
                  padx=5, pady=5,
                  sticky="nsew")

tab1.rowconfigure(8, weight=1)

for i in range(4):
    tab1.columnconfigure(i, weight=1)

# Tab 2 File Processor

Perform sentiment analysis on data from selected CSV file.
Once the user has selected a file, the data is read, processed using the sentiment analysis model, result is displayed in a scrollable widget as negative, neutral, and positive where they add up to 1.

Supported File Formats:
- CSV: Reads text from the review column from the first 50 rows (this can be edited depending on compute power).

Tree view for .csv file on Tab 2

In [15]:
tree = ttk.Treeview(tab2)
tree.grid(row=3, column=0, columnspan=3, sticky="nsew")

# Allow resizing
tab2.rowconfigure(3, weight=1)
for i in range(3):
    tab2.columnconfigure(i, weight=1)

Function for Sentiment Analysis Helper function on Tab 2

In [16]:
df = None
def get_file_sentiment(text):
    encoded = tokenizer(text, padding="max_length", truncation=True, max_length=512, return_tensors="pt")
    output = model(**encoded)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    return {"Negative": scores[0], "Neutral": scores[1], "Positive": scores[2]}

Function to open and read files - Tab 2

In [17]:
def open_files():
    global df
    file_path = tk.filedialog.askopenfilename(filetypes=[("CSV Files","*.csv")])
    if not file_path:
        return
    
    df = pd.read_csv(file_path, encoding="latin-1", index_col=0)

    # Clear existing content
    displayed_file.delete("1.0", "end")

    # Convert DataFrame to string and insert
    displayed_file.insert("end", "Preview (first 500 rows):\n\n")
    displayed_file.insert("end",
                          df.head(500).to_string(index=False, col_space=0)
                         )

In [18]:
l1 = Label(tab2,text="Insert CSV file for processing", font=("Comic Sans MS", 12))
l1.grid(row=2, column=1)

Save function for merged files

In [19]:
def save_file(final_df):
    file_path = filedialog.asksaveasfilename(
        defaultextension=".csv",
        filetypes=[
            ("CSV files", "*.csv")
        ]
    )

    if not file_path:
        return

    if file_path.endswith(".csv"):
        final_df.to_csv(file_path, index=False)

    print("File saved successfully!")

Function for Sentiment Analysis - Tab 2

In [20]:
def run_sentiment_analysis():
    global df

    if df is None:
        print("Error: No file loaded.")
        return

    limited_df = df.head(500)
    results = {}

    # Detect ID column ONCE
    if 'id' in df.columns:
        id_col = 'id'
    elif 'Id' in df.columns:
        id_col = 'Id'
    else:
        id_col = None

    for i, row in tqdm(limited_df.iterrows(), total=len(limited_df)):
        text = row["review"]

        myid = row[id_col] if id_col else i

        scores_dict = get_file_sentiment(text)
        results[myid] = scores_dict

        for key, value in scores_dict.items():
            tab2_display_text.insert(tk.END, f"\n{key}: {value:.4f}\n")

    # Convert results to DataFrame
    results_df = pd.DataFrame.from_dict(results, orient="index")
    results_df = results_df.reset_index().rename(columns={"index": "Id"})
    results_df["Id"] = results_df["Id"].astype(str)

    # Prepare original dataframe
    df_temp = df.copy()

    if id_col:
        df_temp[id_col] = df_temp[id_col].astype(str)
        df_temp = df_temp.set_index(id_col)
    else:
        df_temp.index = df_temp.index.astype(str)

    # Set index for results
    results_df = results_df.set_index("Id")

    # Join
    final_df = results_df.join(df_temp, how="left")
    save_file(final_df)
    final_df = final_df.reset_index()

    print("Join successful!")
    print(final_df.head())

    return final_df

Function to clear/clean display file on Tab 2

In [21]:
def clear_text_file():
    displayed_file.delete('1.0',END)

Function to clear/clean display result/results on Tab 2

In [22]:
def clear_result():
    tab2_display_text.delete('1.0',END)

Display for file on Tab 2

In [23]:
displayed_file = Text(tab2, height=10)
displayed_file.grid(row=3,column=0,columnspan=4, 
                    padx=5,
                    sticky="nsew")
tab2.rowconfigure(3, weight=0)
for i in range(4):
    tab2.columnconfigure(i, weight=1)

scroll = tk.Scrollbar(tab2, command=displayed_file.yview)
scroll.grid(row=3, column=4, sticky="ns")

displayed_file.config(yscrollcommand=scroll.set)

Result Display Screen for Tab 2

In [24]:
tab2_display_text = Text(tab2,height=20)
tab2_display_text.grid(row=6,column=0,columnspan=4,
                       padx=5,
                       sticky="sew")
tab2.rowconfigure(6, weight=0)
for i in range(4):
    tab2.columnconfigure(i, weight=1)

scroll = tk.Scrollbar(tab2, command=displayed_file.yview)
scroll.grid(row=6, column=4, sticky="ns")

tab2_display_text.config(yscrollcommand=scroll.set)

Buttons for opening file Tab 2

In [25]:
button_tab_1 = Button(tab2,text="Open File",width=20,bg="green2",fg="snow",command=open_files)
button_tab_1.grid(row=4,column=0, 
             padx=10,pady=20,
             sticky="w")
button_tab_1.config(font=("Comic Sans MS", 15))

Buttons for get file sentiment file Tab 2

In [26]:
button_tab_2 = Button(tab2,text="Get File Sentiment",width=20,bg="green2",fg="snow",command=run_sentiment_analysis)
button_tab_2.grid(row=4,column=1, 
             padx=10,pady=20,
             sticky="w")
button_tab_2.config(font=("Comic Sans MS", 15))

Buttons for clear display file Tab 2

In [27]:
button_tab_3 = Button(tab2,text="Clear File",width=20,bg="green2",fg="snow",command=clear_text_file)
button_tab_3.grid(row=4,column=2, 
             padx=10,pady=20,
             sticky="w")
button_tab_3.config(font=("Comic Sans MS", 15))

Buttons for clear result Tab 2

In [28]:
button_tab_4 = Button(tab2,text="Clear Result",width=20,bg="green2",fg="snow",command=clear_result)
button_tab_4.grid(row=4,column=3, 
             padx=10,pady=20,
             sticky="w")
button_tab_4.config(font=("Comic Sans MS", 15))

# Help Tab

In [29]:
Help_info_1 = Label(tab3,text="Performs sentiment analysis on the first 50 rows, this can be changed depending on preference.",
                  font=("Comic Sans MS",12),padx=5,pady=5)
Help_info_2 = Label(tab3,text="Ensure the file format is '.csv'.",
                  font=("Comic Sans MS",12),padx=5,pady=5)
Help_info_3 = Label(tab3,text="Ensure there is an 'id' tab for joining to occur.",
                  font=("Comic Sans MS",12),padx=5,pady=5)
Help_info_4 = Label(tab3,text="Ensure the file review column has the review on which sentiment analysis is to be performed.",
                  font=("Comic Sans MS",12),padx=5,pady=5)
Help_info_1.grid(column=0,row=2,sticky="w")
Help_info_2.grid(column=0,row=3,sticky="w")
Help_info_3.grid(column=0,row=4,sticky="w")
Help_info_4.grid(column=0,row=5,sticky="w")

Windows mainloop 

In [31]:
window.mainloop()